# NYC Air Quality PySpark & Databricks Analytics Project
### Dataset: NYC OpenData DOHMH Air Quality (18,862 records)
**Local Path:** `C:\Users\LENOVO\Downloads\Air_Quality.csv`  
**Video Reference:** [Databricks PySpark Project Tutorial (uz3LybMH7II)](https://www.youtube.com/watch?v=uz3LybMH7II)

---
### Project Milestones:
- **[4:12] Dataset Load:** Reading raw NYC CSV, schema inference, and column standardization
- **[6:17] PySpark DataFrame:** Renaming columns to snake_case, type casting (`data_value` to double, `start_date` to DateType), and extracting Year
- **[18:55] Project Implementation & 3 Analytical Objectives / KPIs:**
  - **Objective 1:** Longitudinal Multi-Year Pollutant Trends (2008–2022 Decadal Decline)
  - **Objective 2:** Geographic Disparity & Borough Hotspot Ranking (Manhattan vs. Bronx vs. CDs)
  - **Objective 3:** Public Health Burden & Asthma Emergency Department Morbidity Nexus
- **[41:15] Dashboard Visualization:** Interactive publication-ready charts inside VS Code

In [ ]:
# 1. PySpark Initialization
import os
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_date, year, avg, count, round as sp_round, desc

os.environ.pop("SPARK_HOME", None)
spark = SparkSession.builder.appName("NYC_AirQuality_VSCode").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
print(f"Spark Session Active: Version {spark.version}")

## [4:12] Milestone 1: Ingesting the NYC Dataset
Loading the 18,862 rows from `C:\Users\LENOVO\Downloads\Air_Quality.csv`.

In [ ]:
dataset_path = r"C:\Users\LENOVO\Downloads\Air_Quality.csv"
if not os.path.exists(dataset_path):
    dataset_path = "../data/Air_Quality.csv"

raw_df = spark.read.option("header", "true").csv(dataset_path)
print(f"Total raw records loaded: {raw_df.count():,}")
raw_df.printSchema()

## [6:17] Milestone 2: PySpark DataFrame Cleaning & Type Casting
- Standardizing to clean `snake_case` column identifiers
- Casting `data_value` to `DoubleType`
- Parsing `start_date` to `DateType` and extracting `year`

In [ ]:
col_map = {
    "Unique ID": "unique_id",
    "Indicator ID": "indicator_id",
    "Name": "indicator_name",
    "Measure": "measure",
    "Measure Info": "measure_info",
    "Geo Type Name": "geo_type_name",
    "Geo Join ID": "geo_join_id",
    "Geo Place Name": "geo_place_name",
    "Time Period": "time_period",
    "Start_Date": "start_date_raw",
    "Data Value": "data_value",
    "Message": "message"
}
df_clean = raw_df
for old_c, new_c in col_map.items():
    if old_c in df_clean.columns:
        df_clean = df_clean.withColumnRenamed(old_c, new_c)

df_clean = (
    df_clean
    .withColumn("data_value", col("data_value").cast("double"))
    .withColumn("start_date", to_date(col("start_date_raw"), "MM/dd/yyyy"))
    .withColumn("year", year(col("start_date")))
    .filter(col("data_value").isNotNull())
)
df_clean.createOrReplaceTempView("nyc_air_quality")
print(f"Curated valid rows: {df_clean.count():,}")
df_clean.select("unique_id", "indicator_name", "geo_type_name", "geo_place_name", "year", "data_value").show(5)

## [18:55] Milestone 3: Project Implementation & KPI Development
### Objective 1: Longitudinal Decadal Pollutant Trend (2008 - 2022)

In [ ]:
trends_df = (
    df_clean
    .filter(col("indicator_name").isin("Nitrogen dioxide (NO2)", "Fine particles (PM 2.5)") & (col("geo_type_name") == "Citywide"))
    .groupBy("year", "indicator_name")
    .agg(sp_round(avg("data_value"), 2).alias("annual_mean_value"))
    .orderBy("indicator_name", "year")
)
trends_df.show(20)

### Objective 2: Spatial Disparity & Borough Hotspot Ranking

In [ ]:
borough_df = (
    df_clean
    .filter(col("indicator_name").isin("Nitrogen dioxide (NO2)", "Fine particles (PM 2.5)") & (col("geo_type_name") == "Borough"))
    .groupBy("geo_place_name", "indicator_name")
    .agg(sp_round(avg("data_value"), 2).alias("mean_concentration"))
    .orderBy("indicator_name", desc("mean_concentration"))
)
borough_df.show()

### Objective 3: Public Health Burden (Asthma Emergency Visits by Borough)

In [ ]:
asthma_df = (
    df_clean
    .filter(col("indicator_name").like("%Asthma%") & (col("geo_type_name") == "Borough"))
    .groupBy("geo_place_name")
    .agg(sp_round(avg("data_value"), 2).alias("avg_asthma_er_rate_per_100k"))
    .orderBy(desc("avg_asthma_er_rate_per_100k"))
)
asthma_df.show()

## [41:15] Milestone 4: Visualizations in VS Code

In [ ]:
# Plot 1: Decadal Decline Trend
sns.set_theme(style="whitegrid")
t_pdf = trends_df.toPandas()
plt.figure(figsize=(10, 5))
for ind in t_pdf["indicator_name"].unique():
    sub = t_pdf[t_pdf["indicator_name"] == ind]
    plt.plot(sub["year"], sub["annual_mean_value"], marker="o", linewidth=2.5, label=ind)
plt.title("Objective 1: NYC Annual Air Pollutant Decline (2008 - 2022)", fontsize=13, fontweight="bold")
plt.xlabel("Year")
plt.ylabel("Mean Concentration (NO2: ppb, PM2.5: µg/m³)")
plt.legend()
plt.show()

In [ ]:
# Plot 2: Borough Comparison
b_pdf = borough_df.toPandas()
plt.figure(figsize=(10, 5))
sns.barplot(data=b_pdf, x="geo_place_name", y="mean_concentration", hue="indicator_name", palette=["#20639b", "#ed553b"])
plt.title("Objective 2: Borough Comparison (NO2 vs. PM2.5)", fontsize=13, fontweight="bold")
plt.xlabel("Borough")
plt.ylabel("Mean Concentration")
plt.show()

In [ ]:
# Plot 3: Asthma Morbidity Disparity
a_pdf = asthma_df.toPandas()
plt.figure(figsize=(9, 5))
sns.barplot(data=a_pdf, x="geo_place_name", y="avg_asthma_er_rate_per_100k", hue="geo_place_name", palette="rocket", legend=False)
plt.title("Objective 3: Asthma Emergency Rate Disparity (per 100k)", fontsize=13, fontweight="bold")
plt.xlabel("Borough")
plt.ylabel("Asthma ER Rate per 100k Population")
plt.show()